## Import and Clean Dataset

### 1. Import libraries

We import the libraries needed to request data from the API,
organize the data, randomly select positions, and add pauses between requests.

In [1]:
import requests
import pandas as pd
import random
import time

### 2. Set up the API and sampling plan

We use the Hugging Face API to collect 30,000 conversations.
We select random positions in the dataset and download up to 30 conversations
from each position. A random seed makes the selection reproducible.

In [2]:
# Hugging Face API address
url = "https://datasets-server.huggingface.co/rows"

# Sampling settings
total_rows = 837989
batch_size = 30
target_rows = 30000

# Set a seed so the same positions can be selected again
random.seed(42)

# Calculate how many batches are needed
number_of_batches = (target_rows + batch_size - 1) // batch_size

# Randomly select non-overlapping batches
batch_ids = random.sample(
    range(total_rows // batch_size),
    number_of_batches
)

print("Number of batches:", number_of_batches)

Number of batches: 1000


### 3. Download conversations

We download conversations from the selected positions in the dataset.
We pause between requests to reduce the chance of sending too many requests.
If a request fails, we retry it and record any batches that still fail.

In [3]:
# Create empty lists to store conversations and failed batches
all_rows = []
failed_batches = []

# Go through each randomly selected batch
for i, batch_id in enumerate(batch_ids):

    # Calculate the starting position and number of rows
    offset = batch_id * batch_size
    length = min(batch_size, target_rows - i * batch_size)

    # Set the information needed for the API request
    params = {
        "dataset": "allenai/WildChat-1M",
        "config": "default",
        "split": "train",
        "offset": offset,
        "length": length
    }

    success = False

    # Try each request up to three times
    for attempt in range(3):
        try:
            # Request data from the API
            response = requests.get(url, params=params, timeout=30)

            # Wait longer if the server reports too many requests
            if response.status_code == 429:
                print(f"Batch {i + 1}: too many requests. Waiting 60 seconds...")
                time.sleep(60)
                continue

            # Check whether the request was successful
            response.raise_for_status()

            # Extract the rows from the API response
            rows = response.json()["rows"]

            # Add each conversation to all_rows
            for item in rows:
                all_rows.append(item["row"])

            success = True
            break

        except requests.exceptions.RequestException as e:
            # Show the error and wait before trying again
            print(f"Batch {i + 1}, attempt {attempt + 1} failed: {e}")
            time.sleep(5)

    # Save the starting position if all attempts failed
    if not success:
        failed_batches.append(offset)

    # Show progress every 20 batches
    if (i + 1) % 20 == 0 or i == number_of_batches - 1:
        print(f"Completed {i + 1}/{number_of_batches} batches")

    # Pause before sending the next request
    time.sleep(2)

Completed 20/1000 batches
Batch 31: too many requests. Waiting 60 seconds...
Completed 40/1000 batches
Completed 60/1000 batches
Batch 61: too many requests. Waiting 60 seconds...
Completed 80/1000 batches
Batch 92: too many requests. Waiting 60 seconds...
Completed 100/1000 batches
Batch 119: too many requests. Waiting 60 seconds...
Completed 120/1000 batches
Completed 140/1000 batches
Batch 149: too many requests. Waiting 60 seconds...
Completed 160/1000 batches
Completed 180/1000 batches
Batch 190: too many requests. Waiting 60 seconds...
Completed 200/1000 batches
Completed 220/1000 batches
Batch 221: too many requests. Waiting 60 seconds...
Completed 240/1000 batches
Batch 252: too many requests. Waiting 60 seconds...
Batch 260, attempt 1 failed: HTTPSConnectionPool(host='datasets-server.huggingface.co', port=443): Read timed out. (read timeout=30)
Completed 260/1000 batches
Completed 280/1000 batches
Batch 286: too many requests. Waiting 60 seconds...
Completed 300/1000 batches
B

### 4. Create and check the DataFrame

We put the downloaded conversations into a pandas DataFrame.
Then, we check the number of conversations and any failed batches.

In [4]:
# Convert the downloaded conversations into a DataFrame
df = pd.DataFrame(all_rows)

# Check the results
print("Conversations downloaded:", len(df))
print("Failed batch offsets:", failed_batches)

# Display the first five rows
df.head()

Conversations downloaded: 30000
Failed batch offsets: []


,conversation_hash,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted,state,country,hashed_ip,header
0,e0c5aec012f5acde923390707b58741f,gpt-4-1106-preview,2023-12-23T17:49:33Z,[{'content': 'Create a scam-baiting response t...,3,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00013213619240559638, '...",False,True,Florida,United States,96468a552ddf93221173da70c4307bdd7039d76e2ec154...,"{'accept-language': None, 'user-agent': 'Mozil..."
1,44e3256dff0f1ed796130746df4dc155,gpt-3.5-turbo-0613,2023-12-23T17:50:21Z,"[{'content': 'Will this formula work, and what...",3,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0007662957068532705, 'i...",False,False,Oxfordshire,United Kingdom,e0f8f253fd649edfd6f2afc4e990d1d779321a306c3cb7...,"{'accept-language': 'en-US,en;q=0.9', 'user-ag..."
2,962abf7e81fcf713de545607b23ef599,gpt-3.5-turbo-0613,2023-12-23T17:51:21Z,"[{'content': '– Нет, Оксана Игоревна, – прогов...",1,Russian,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0012769330060109496, 'i...",False,False,Chelyabinsk Oblast,Russia,2eef7d0f0b5b3cdd22f40849eefe0f13b05730cffa5bfd...,"{'accept-language': 'ru-RU,ru;q=0.9,en-US;q=0...."
3,7998f2e098c55ab13a0cf3462a7fae98,gpt-3.5-turbo-0613,2023-12-23T17:52:12Z,"[{'content': '– Нет, Оксана Игоревна, – прогов...",1,Russian,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0012769330060109496, 'i...",False,False,Chelyabinsk Oblast,Russia,2eef7d0f0b5b3cdd22f40849eefe0f13b05730cffa5bfd...,"{'accept-language': 'ru-RU,ru;q=0.9,en-US;q=0...."
4,0258a1739483b7e196b8418c222af34e,gpt-3.5-turbo-0613,2023-12-23T17:52:41Z,"[{'content': '– Нет, Оксана Игоревна, – прогов...",1,Russian,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0012769330060109496, 'i...",False,False,Chelyabinsk Oblast,Russia,2eef7d0f0b5b3cdd22f40849eefe0f13b05730cffa5bfd...,"{'accept-language': 'ru-RU,ru;q=0.9,en-US;q=0...."


### 5. Filter conversations

We keep English conversations with exactly five non-empty user messages.
This ensures that each conversation has enough user messages for analysis.

In [5]:
# Count non-empty user messages in a conversation
def count_user_messages(conversation):
    count = 0

    for message in conversation:
        content = message.get("content")

        if message.get("role") == "user" and isinstance(content, str) and content.strip():
            count += 1

    return count


# Keep conversations labeled as English
english_df = df[df["language"] == "English"].copy()

# Count user messages in each English conversation
english_df["user_message_count"] = english_df["conversation"].apply(
    count_user_messages
)

# Keep conversations with exactly five non-empty user messages
filtered_df = english_df[english_df["user_message_count"] == 5].copy()

# Check how many conversations remain after each step
print("Original conversations:", len(df))
print("English conversations:", len(english_df))
print("English conversations with exactly 5 user messages:", len(filtered_df))

# Display the first five rows
filtered_df[["conversation_hash", "language", "user_message_count"]].head()

Original conversations: 30000
English conversations: 17074
English conversations with exactly 5 user messages: 433


,conversation_hash,language,user_message_count
34,93178785faa50907bd35da0f7c03adf5,English,5
91,ff8c1c415320ff168553be0d5d835a6a,English,5
102,c5b720b4e66eb36ef0e5c953aae366ac,English,5
164,1b7761de9c542637593cd656c24e2e1b,English,5
392,5d5c1987abca530b3bd62079ce66ace6,English,5


### 6. Check for duplicates

We check whether conversation IDs or complete conversation contents
are repeated before deciding how to handle duplicates.

In [6]:
import json

# Count repeated conversation IDs
duplicate_ids = filtered_df["conversation_hash"].duplicated().sum()

# Convert each conversation to a string so it can be compared
conversation_text = filtered_df["conversation"].apply(
    lambda conversation: json.dumps(conversation, sort_keys=True)
)

# Count exact duplicates across all message fields
duplicate_conversations = conversation_text.duplicated().sum()

print("Repeated conversation IDs:", duplicate_ids)
print("Exact duplicate conversations:", duplicate_conversations)

Repeated conversation IDs: 0
Exact duplicate conversations: 0


### 7. Check for repeated user messages

We compare the sequence of user messages in each conversation.
This helps us find conversations with the same user text,
even if other information is different.

In [7]:
# Extract the sequence of non-empty user messages
def get_user_messages(conversation):
    messages = []

    for message in conversation:
        content = message.get("content")

        if message.get("role") == "user" and isinstance(content, str) and content.strip():
            messages.append(content.strip())

    return tuple(messages)


# Create a column containing the user messages
filtered_df["user_messages"] = filtered_df["conversation"].apply(
    get_user_messages
)

# Count conversations with the same sequence of user messages
duplicate_user_text = filtered_df["user_messages"].duplicated().sum()

print("Conversations with repeated user text:", duplicate_user_text)

Conversations with repeated user text: 1


### 8. Remove repeated user messages

For the main analysis, we keep the first conversation from each set
with the same sequence of user messages. We also keep the original
filtered data so we can compare results later.

In [8]:
# Keep the first conversation with each sequence of user messages
analysis_df = filtered_df.drop_duplicates(
    subset="user_messages",
    keep="first"
).copy()

# Compare the number of conversations before and after removal
print("Before removing repeated user text:", len(filtered_df))
print("After removing repeated user text:", len(analysis_df))
print("Conversations removed:", len(filtered_df) - len(analysis_df))

Before removing repeated user text: 433
After removing repeated user text: 432
Conversations removed: 1


### 9. Extract the first and last user messages             
### NOT RUN

We extract the first and last non-empty user messages from each conversation.
These messages will be used for the sentiment comparison.

In [9]:
# # Extract the first user message
# analysis_df["first_message"] = analysis_df["user_messages"].apply(
#     lambda messages: messages[0]
# )

# # Extract the last user message
# analysis_df["last_message"] = analysis_df["user_messages"].apply(
#     lambda messages: messages[-1]
# )

# # Check the first five conversations
# analysis_df[
#     ["conversation_hash", "user_message_count", "first_message", "last_message"]
# ].head()

## 10. Remove conversations containing possible API keys

Some conversations may contain text that looks like an OpenAI API key. We remove any conversation containing `sk-proj-` to avoid sharing possible keys in the dataset. We also record the number of conversations removed.

In [10]:
# Remove conversations that contain a possible OpenAI project API key
# Check each conversation for a possible OpenAI project API key
contains_key = analysis_df.apply(
    lambda row: "sk-proj-" in row.to_json(force_ascii=False),
    axis=1
)

# Count the conversations containing a possible key
print("Conversations removed:", contains_key.sum())

# Keep only conversations without this pattern
analysis_df = analysis_df.loc[~contains_key].copy()

# Show the number of conversations remaining
print("Conversations remaining:", len(analysis_df))

Conversations removed: 0
Conversations remaining: 432


### 11. Save the cleaned data

We select the columns needed for analysis and save the cleaned data
as a JSON file so team members can use the same data.

In [11]:
# Select the columns needed for analysis
cleaned_data = analysis_df[
    ["conversation", "user_messages", "user_message_count"]
].copy()

# Save the cleaned data as a new JSON file
cleaned_data.to_json(
    "wildchat_cleaned_exactly5.json",
    orient="records",
    force_ascii=False,
    indent=2
)

# Check the saved data
print("Cleaned data saved to wildchat_cleaned_exactly5.json")
print("Number of conversations:", len(cleaned_data))
print("Columns:", cleaned_data.columns.tolist())

Cleaned data saved to wildchat_cleaned_exactly5.json
Number of conversations: 432
Columns: ['conversation', 'user_messages', 'user_message_count']


### If wanna to use the cleaned data later u can use the code below:


In [12]:
import pandas as pd
analysis_df = pd.read_json("wildchat_cleaned_exactly5.json")